In [ ]:
# | default_exp soil_params

In [ ]:
# | hide
from fastcore import *
from nbdev.showdoc import *
from pprint import pprint

In [ ]:
# | export
from plant_hydraulics.parameter_classes import Soil

### Root Distribution by Soil Layer


Gale, M. R., & Grigal, D. F. (1987). Vertical root distributions of northern 
tree species in relation to successional status. Canadian Journal of Forest Research, 
17(8), 829-834

The root distribution follows the model of Gale and Grigal (1987), where the 
cumulative fraction of roots above depth d (in cm) is given by:

Y(d) = 1 − β^d

The fraction of roots within a layer between depths z1 and z2 is then:

rootfr = Y(z2) − Y(z1) = β^z1 − β^z2


Root fraction calculated using Y(d) = 1 − β^d with β = 0.90 (shallow root profile).

| Layer | dz (m) | z1 (cm) | z2 (cm) | rootfr | Cumulative rootfr |
|------:|-------:|--------:|--------:|-------:|------------------:|
|     0 |   0.05 |     0.0 |     5.0 |  0.410 |             0.410 |
|     1 |   0.05 |     5.0 |    10.0 |  0.242 |             0.651 |
|     2 |   0.10 |    10.0 |    20.0 |  0.227 |             0.878 |
|     3 |   0.10 |    20.0 |    30.0 |  0.079 |             0.957 |
|     4 |   0.20 |    30.0 |    50.0 |  0.031 |             0.988 |
|     5 |   0.20 |    50.0 |    70.0 |  0.008 |             0.995 |
|     6 |   0.20 |    70.0 |    90.0 |  0.003 |             0.998 |
|     7 |   0.30 |    90.0 |   120.0 |  0.001 |             0.999 |
|     8 |   0.40 |   120.0 |   160.0 |  0.000 |             1.000 |
|     9 |   0.40 |   160.0 |   200.0 |  0.000 |             1.000 |
|    10 |   0.50 |   200.0 |   250.0 |  0.000 |             1.000 |


In [ ]:
# | export


def soil_params(soil: Soil) -> Soil:
    """
    Set soil hydraulic parameters, soil depth, and rooting fraction.

    __Parameters:__

    - soil: Soil object with the following input attribute:

        - texture: Soil texture class:
                1: sand, 2: loamy sand, 3: sandy loam, 4: silt loam,
                5: loam, 6: sandy clay loam, 7: silty clay loam,
                8: clay loam, 9: sandy clay, 10: silty clay, 11: clay

    __Returns:__

    - soil: Updated Soil object with the following attributes:

        - nlevsoi: Number of soil layers.
        - dz: Soil layer thickness (m).
        - rootfr: Fraction of roots in each soil layer (-).
        - watsat: Soil layer volumetric water content at saturation, i.e. porosity (-).
        - psisat: Soil layer matric potential at saturation (mm).
        - hksat: Soil layer hydraulic conductivity at saturation (mm H2O/s).
        - bsw: Soil layer Clapp and Hornberger "b" parameter (-).

    Parameters
    ----------
    soil : Soil
        Soil object with the following input attribute:
        - texture : int
            Soil texture class:
                1: sand, 2: loamy sand, 3: sandy loam, 4: silt loam,
                5: loam, 6: sandy clay loam, 7: silty clay loam,
                8: clay loam, 9: sandy clay, 10: silty clay, 11: clay

    Returns
    -------
    soil : Soil
        Updated soil object with the following attributes:
        - nlevsoi : int
            Number of soil layers.
        - dz : list[float]
            Soil layer thickness (m).
        - rootfr : list[float]
            Fraction of roots in each soil layer (-).
        - watsat : list[float]
            Soil layer volumetric water content at saturation, i.e. porosity (-).
        - psisat : list[float]
            Soil layer matric potential at saturation (mm).
        - hksat : list[float]
            Soil layer hydraulic conductivity at saturation (mm H2O/s).
        - bsw : list[float]
            Soil layer Clapp and Hornberger "b" parameter (-).
    """
    # Clapp, R. B., & Hornberger, G. M. (1978). Empirical equations for some soil
    # hydraulic properties. Water resources research, 14(4), 601-604.

    # Volumetric water content at saturation (porosity)
    # AKA voids between solid particles
    theta_sat = [
        0.395,
        0.410,
        0.435,
        0.485,
        0.451,
        0.420,
        0.477,
        0.476,
        0.426,
        0.492,
        0.482,
    ]

    # Matric potential at saturation (mm)
    # Think of it like a sponge: a dry sponge exerts strong suction on any water
    # it contacts (very negative matric potential), while a fully saturated
    # sponge exerts almost no suction (matric potential near zero).
    psi_sat = [-121, -90, -218, -786, -478, -299, -356, -630, -153, -490, -405]

    # Clapp and Hornberger "b" parameter
    b_param = [4.05, 4.38, 4.90, 5.30, 5.39, 7.12, 7.75, 8.52, 10.40, 10.40, 11.40]

    # Hydraulic conductivity at saturation
    k_sat = [
        1.056,
        0.938,
        0.208,
        0.0432,
        0.0417,
        0.0378,
        0.0102,
        0.0147,
        0.0130,
        0.0062,
        0.0077,
    ]

    # Convert k_sat cm/min -> mm/s
    k_sat = [each_k * 10.0 / 60.0 for each_k in k_sat]

    # Soil layer thickness (total depth is 2.5 m)
    soil.nlevsoi = 11
    soil.dz = [0.05, 0.05, 0.10, 0.10, 0.20, 0.20, 0.20, 0.30, 0.40, 0.40, 0.50]

    # Root profile --------------------------------------------------------------
    # Root distribution follows the model of Gale and Grigal (1987)

    # root profile parameter: shallow profile
    beta_param = 0.90

    # root profile parameter: deep profile
    # beta_param = 0.97;

    soil.rootfr = []
    z2 = 0.0
    for each_soil_layer in range(soil.nlevsoi):
        
        if each_soil_layer == 0:
            
            # Cumulative depth from the soil surface in centimeters
            z2 = soil.dz[each_soil_layer] * 100.0
            soil.rootfr.append(1.0 - beta_param**z2)
        
        else:
            z1 = z2
            z2 = z1 + soil.dz[each_soil_layer] * 100.0
            soil.rootfr.append(beta_param**z1 - beta_param**z2)

    # Soil texture to process (0-indexed)
    k = soil.texture - 1

    # Soil layer values
    soil.watsat = [theta_sat[k]] * soil.nlevsoi
    soil.psisat = [psi_sat[k]] * soil.nlevsoi
    soil.hksat = [k_sat[k]] * soil.nlevsoi
    soil.bsw = [b_param[k]] * soil.nlevsoi

    return soil

### Example soil_params()

In [ ]:
soil = Soil()
pprint(vars(soil_params(soil=soil)))

{'bsw': [5.39, 5.39, 5.39, 5.39, 5.39, 5.39, 5.39, 5.39, 5.39, 5.39, 5.39],
 'dz': [0.05, 0.05, 0.1, 0.1, 0.2, 0.2, 0.2, 0.3, 0.4, 0.4, 0.5],
 'h2osoi_vol': [],
 'hksat': [0.0069500000000000004,
           0.0069500000000000004,
           0.0069500000000000004,
           0.0069500000000000004,
           0.0069500000000000004,
           0.0069500000000000004,
           0.0069500000000000004,
           0.0069500000000000004,
           0.0069500000000000004,
           0.0069500000000000004,
           0.0069500000000000004],
 'nlevsoi': 11,
 'psi': [],
 'psisat': [-478, -478, -478, -478, -478, -478, -478, -478, -478, -478, -478],
 'rootfr': [0.40950999999999993,
            0.24181155989999997,
            0.22710178550943075,
            0.07918549631535311,
            0.03723738306789612,
            0.004527196459102321,
            0.0005504014001719315,
            7.2948102027868e-05,
            3.181514910617433e-06,
            4.7025599470265064e-08,
            7.01871